In [43]:
import os
import torch
import pytorch_lightning as pl
from transformers import get_scheduler, AutoModelForCausalLM, AutoProcessor
from florence2_large import processing_florence2
from peft import LoraConfig, get_peft_model
from pytorch_lightning.loggers.wandb import WandbLogger
from pytorch_lightning import Trainer
from torch.utils.data import DataLoader, Dataset
from datetime import datetime
import yaml
import wandb
from checkpoint_callback import CustomModelCheckpoint
from peft import PeftModel, PeftConfig
from transformers import AutoConfig
import supervision as sv
import pandas as pd
import numpy as np
import ast

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
# loads config files for the model settings or other features
def load_config(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

<h2>Data Preprocessing</h2>

In [33]:
# csv file with padchest's metadata
# TODO: change if source files are moved
df = pd.read_csv("../padchest_labels.csv")

# drop all floats in the labels column
# some of the labels are given as 'nan'
df = df[~df['Labels'].apply(lambda x: isinstance(x, float))]

df.reset_index(drop=True, inplace=True)

# gets the unique labels in the dataset
unique_labels = set()
for labels in df['Labels']:
    # labels are stored as strings, need to convert to a list
    label_list = eval(labels)
    label_list = [l.strip(' ') for l in label_list]  # removes spaces in labels list
    unique_labels.update(label_list)

unique_labels = sorted(unique_labels)
del(unique_labels[0])  # a space is listed as a label, removes it
print(unique_labels)
print(f"There are {len(unique_labels)} unique labels in the dataset")

CLASSES = [c.lower() for c in unique_labels]

# for c in CLASSES:
#     print(c)

['COPD signs', 'Chilaiditi sign', 'NSG tube', 'abnormal foreign body', 'abscess', 'adenopathy', 'air bronchogram', 'air fluid level', 'air trapping', 'alveolar pattern', 'aortic aneurysm', 'aortic atheromatosis', 'aortic button enlargement', 'aortic elongation', 'aortic endoprosthesis', 'apical pleural thickening', 'artificial aortic heart valve', 'artificial heart valve', 'artificial mitral heart valve', 'asbestosis signs', 'ascendent aortic elongation', 'atelectasis', 'atelectasis basal', 'atypical pneumonia', 'axial hyperostosis', 'azygoesophageal recess shift', 'azygos lobe', 'blastic bone lesion', 'bone cement', 'bone metastasis', 'breast mass', 'bronchiectasis', 'bronchovascular markings', 'bullas', 'calcified adenopathy', 'calcified densities', 'calcified fibroadenoma', 'calcified granuloma', 'calcified mediastinal adenopathy', 'calcified pleural plaques', 'calcified pleural thickening', 'callus rib fracture', 'cardiomegaly', 'catheter', 'cavitation', 'central vascular redistrib

<h2>Model</h2>

In [22]:
class FlorenceLightningModel(pl.LightningModule):
    def __init__(self, model, processor, lr=1e-6, num_training_steps=None):
        super(FlorenceLightningModel, self).__init__()
        self.model = model
        self.processor = processor
        self.lr = float(lr)
        self.num_training_steps = num_training_steps
        self.test_outputs = []
        self.valid_outputs = []

    def training_step(self, batch, batch_idx):
        images,questions,answers, tasks = batch
        inputs = self.processor(
            text=questions, 
            images=images, 
            return_tensors="pt", 
            padding=True
        ).to(self.device)
        input_ids = inputs["input_ids"]
        pixel_values = inputs["pixel_values"]
        labels = self.processor.tokenizer(
            text=answers,
            return_tensors="pt",
            padding=True,
            return_token_type_ids=False
        ).input_ids.to(self.device)

        outputs = self.model(input_ids=input_ids, pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, batch_size = len(images), sync_dist=True)  # Log to progress bar
        torch.cuda.empty_cache()  
        return loss

    def validation_step(self, batch, batch_idx):
        images,questions,answers, tasks = batch
        # import pdb; pdb.set_trace()
        inputs = self.processor( 
            text=questions, 
            images=images, 
            return_tensors="pt", 
            padding=True
        ).to(self.device)
        from tools import evluate_resultsevluate_results
        batch_results = evluate_resultsevluate_results(model=self.model, inputs=inputs,processor=self.
                                                       processor, answers=answers, images=images,batch_idx=batch_idx,
                                                       questions=questions)
        batch_data = [
            wandb.Image(img, caption=cap) 
            for img,cap in zip(batch_results['res_samples'], batch_results['captions'])
        ]
        self.logger.experiment.log({
            "generated_images": batch_data,
            "epoch": self.current_epoch,  
            "batch_idx": batch_idx,      
            "step": self.current_epoch  
        })
        self.valid_outputs.append(batch_results)

        return batch_results

    def on_validation_epoch_end(self):
        all_predictions = []
        all_targets = []

        for batch_result in self.valid_outputs:
            all_predictions.extend(batch_result["predictions"])
            all_targets.extend(batch_result["targets"])

        confusion_matrix = sv.ConfusionMatrix.from_detections(
            predictions=all_predictions, 
            targets=all_targets, 
            classes=CLASSES
        )

        mean_average_precision = sv.MeanAveragePrecision.from_detections(
            predictions=all_predictions, 
            targets=all_targets
        )

        self.log("val/mAP_50_95", mean_average_precision.map50_95)
        self.log("val/mAP_50", mean_average_precision.map50)
        self.log("val/mAP_75", mean_average_precision.map75)

    def test_step(self, batch, batch_idx):
        images,questions,answers, tasks = batch
        # import pdb; pdb.set_trace()
        inputs = self.processor( 
            text=questions, 
            images=images, 
            return_tensors="pt", 
            padding=True
        ).to(self.device)

        from tools import evluate_resultsevluate_results
        batch_results  = evluate_resultsevluate_results(model=self.model, inputs=inputs,processor=self.processor, answers=answers, images=images,batch_idx=batch_idx,questions=questions)
        batch_data = [
            wandb.Image(img, caption) 
            for img,caption in zip(batch_results['res_samples'], batch_results['captions'])
        ]
        self.logger.experiment.log({
            "generated_images": batch_data,
            "epoch": self.current_epoch,  
            "batch_idx": batch_idx,      
            "step": self.current_epoch  
        })
        self.test_outputs.append(batch_results)

        return batch_results
        
    def on_test_epoch_end(self):
        all_predictions = []
        all_targets = []

        for batch_result in self.test_outputs:
            all_predictions.extend(batch_result["predictions"])
            all_targets.extend(batch_result["targets"])
        
        confusion_matrix = sv.ConfusionMatrix.from_detections(
            predictions=all_predictions, 
            targets=all_targets, 
            classes=CLASSES
        )

        mean_average_precision = sv.MeanAveragePrecision.from_detections(
            predictions=all_predictions, 
            targets=all_targets
        )

        print("mAP_50_95:", mean_average_precision.map50_95)
        print("mAP_50:", mean_average_precision.map50)
        print("mAP_75:", mean_average_precision.map75)
        self.log("test/mAP_50_95", mean_average_precision.map50_95)
        self.log("test/mAP_50", mean_average_precision.map50)
        self.log("test/mAP_75", mean_average_precision.map75)

        print("Confusion Matrix:\n", confusion_matrix.matrix)
        print("Mean Average Precision:\n", mean_average_precision)

    def configure_optimizers(self):
        print('self.lr:', self.lr)
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.lr)
        lr_scheduler = get_scheduler(
            name="linear",
            optimizer=optimizer,
            num_warmup_steps=0,
            num_training_steps=self.num_training_steps,
        )
        return [optimizer], [lr_scheduler]

<h2>Running Model</h2>

In [54]:
# loads the cofig for running the model
config_path = "configs/experiment.yaml"
config = load_config(config_path)

<h4>Initialising Model</h4>

In [55]:
# uses GPU if able
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# state version of model if needed
# REVISION = 

MODEL_NAME = "microsoft/Florence-2-large"

### initialising the model
# downloads the model config from hugging face 
config_model = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
config_model.vision_config.model_type = "davit"
# model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code = True, config = config_model,revision = REVISION).to(DEVICE)
# builds generic model using florence2
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code = True, config = config_model).to(DEVICE)
# defines the processor to be used (florence2)
processor = processing_florence2.Florence2Processor.from_pretrained("./florence2_large")
processor.image_processor.size = config['model']['processor']['image_size']
processor.image_processor.crop_size = config['model']['processor']['crop_size']

<h4>Fine Tuning Parameters Using PEFT</h4>

In [56]:
if config['model']['peft']['use_peft']:
    # load an existing peft model checkpoint if there is one
    if config['model']['peft']['lora_checkpoint'] not in [None, "False"]:
        lora_checkpoint = LoraConfig.from_pretrained(config['model']['peft']['lora_checkpoint'])
        model = PeftModel.from_pretrained(model, lora_checkpoint, is_trainable=True)
    else:
        lora_config = LoraConfig(
                r=8,
                lora_alpha=8,
                target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "linear", "Conv2d", "lm_head", "fc2"],
                task_type="CAUSAL_LM",
                lora_dropout=0.05,
                bias="none",
                inference_mode=False,
                use_rslora=True,
                init_lora_weights="gaussian"
            )
        model = get_peft_model(model, lora_config)

# else, fine tune entire language part, only freeze the vision part
elif config['model']['finetune']:
    for param in model.vision_tower.parameters():
        param.requires_grad = False

# otherwise, train the entire model
else:
    for param in model.parameters():
        param.requires_grad = True

<h4>Loading the Images for the Dataset</h4>

In [57]:
class PadChestDataset(Dataset):
    def __init__(self, csv_path, root, transform=None):
        self.data_info = pd.read_csv(csv_path)
        self.root = root
        self.transform = transform

    # gets an image from a given index
    # does preprocessing on the image too
    def __getitem__(self, index):
        # gets image path and loads the image
        img_path = os.path.join(self.root, self.data_info.iloc[index]['img_path'])
        img_array = np.array(Image.open(img_path))

        # normalise the image
        img_array = (img_array / img_array.max()) * 255
        img = Image.fromarray(img_array.astype('uint8')).convert('RGB')

        # apply transformation
        if self.transform:
            img_np = np.array(img)
            img = self.transform(image = img_np)['image']

        # gets entire row from dataframe and returns as a Series
        labels_str = self.data_info.iloc[index]['Labels']
        labels = ast.literal_eval(labels_str)
        labels = [l.strip(' ') for l in labels]  # removes spaces in labels list
        labels = [label.lower() for label in labels]  # makes labels lowercase

        return {
            "image": img,
            "labels": labels
        }
    
    def __len__(self):
        return len(self.data_info)

In [58]:
# classes that manage the loading of padchest and params for the model
class MultiInstructorPadChest(Dataset):
    def __init__(self, base_dataset, task="<CAPTION_TO_PHRASE_GROUNDING>",task_prompt="Locate the phrases in the caption: {input}.", use_definition=True):
        self.base_dataset = base_dataset
        self.task_prompt = task_prompt
        self.task = task
        # self.scale_factor = 1000
        self.definition = yaml.safe_load(open('configs/padchest_definition.yaml'))
        self.use_definition = use_definition
        print('Using definition:', self.use_definition)

    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        sample = self.base_dataset[idx]
        image = sample['image']
        labels = sample['labels']  # list of strings for each label

        # if using definitions, get descriptions for all labels
        # TODO: write the label defs
        if self.use_definition:
            labels_with_def = [f"{label} means {self.definition.get(label, 'no definition available')}." for label in labels]
            det_obj = " ".join(labels_with_def)
        # no definition, join with commas
        else:
            det_obj = ", ".join(labels)

        # create the task prompt with all labels concatenated
        task_prompt = self.task_prompt.format(input = det_obj)
        task_prompt = self.task + task_prompt

        final_answer = det_obj

        return {
            'image': image,
            'question': task_prompt,
            'answer': final_answer,
            'task': self.task
        }

class PadChestDataLoaderManager(pl.LightningDataModule):
    def __init__(self, config):
        super().__init__()
        self.img_root = config.get("img_root")
        self.annotation_csv = config.get("annotation_csv")
        self.batch_size = config.get("batch_size", 8)
        self.data_pct = config.get("data_pct", 1.0)
        self.num_workers = config.get("num_workers", 0)
        self.device = config.get("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
        self.use_definition = config["use_definition"]

    def collate_fn(self,batch):
        # unzip the batch into questions, answers, and images
        questions = [item['question'] for item in batch]
        answers = [item['answer'] for item in batch]
        images = [item['image'] for item in batch]
        tasks = [item['task'] for item in batch]
    
        return images,questions, answers, tasks
        
    
    # creates a pytorch DataLoader
    # need to specify the split (train/test/validation)
    def create_dataloader(self, split):
        # initialises the base dataset (PadChest)
        base_dataset = PadChestDataset(
            csv_path = self.annotation_csv,
            root = self.img_root,
            transform = None
        )

        # initialises the multi task instructor dataset
        multi_task_dataset = MultiInstructorPadChest(
            base_dataset = base_dataset,
            use_definition = self.use_definition
        )

        # creates and returns the DataLoader
        return DataLoader(
            multi_task_dataset,
            batch_size = self.batch_size,
            collate_fn = self.collate_fn,
            num_workers = self.num_workers,
            shuffle = True if split == 'train' else False  # only shuffle if this is for the training set
        )
    
    def train_dataloader(self):
        return self.create_dataloader(split='train')

    def val_dataloader(self):
        return self.create_dataloader(split='test')

    def test_dataloader(self):
        return self.create_dataloader(split='test')

<h4>Makes the DataLoaders for PadChest</h4>

In [59]:
data_loader_manager = PadChestDataLoaderManager({
    "img_root": config['dataset']['padchest']['img_root'],
    "annotation_csv": config['dataset']['padchest']['annotation_csv'],
    "batch_size": config['trainer']['train_batch_size'],
    "data_pct": config['dataset']['padchest']['data_pct'],
    "num_workers": config['trainer']['num_workers'],
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "processor": None,  # use default processor
    "use_definition": False  # change to True if want to use a definition as a prompt
})

<h4>Trains the DataLoaders and Makes the Model</h4>

In [60]:
train_dataloader = data_loader_manager.train_dataloader()
val_dataloader = data_loader_manager.test_dataloader()

dataset_size = len(train_dataloader.dataset)
num_training_steps = (dataset_size + config['trainer']['train_batch_size'] - 1) // config['trainer']['train_batch_size']

lightning_model = FlorenceLightningModel(model=model, processor=processor, lr=config['trainer']['learning_rate'], num_training_steps=num_training_steps)


Using definition: False
Using definition: False


In [ ]:
if config['trainer']['checkpoint_dir'] is not None:
        os.makedirs(config['trainer']['checkpoint_dir'], exist_ok=True)

custom_checkpoint_callback = CustomModelCheckpoint(
    dirpath = config['trainer']['checkpoint_dir'],
    filename = 'model-{epoch}-{step}',
    save_top_k = 1,  # Save top 2 models based on the monitored metric
    monitor = 'val_loss',  # Monitor a different metric (e.g., val_accuracy)
    mode = 'min',  # Mode for monitoring (min for loss, max for accuracy)
    # every_n_train_steps=500,  # every_n_train_steps >= save_top_k*val_check_interval
    save_embedding_layers = True,  # Save the embedding layers
    verbose = True  # Set to True to log when checkpoints are saved
)

Trainer()

trainer = Trainer(
    max_epochs = config['trainer']['max_epochs'],
    accelerator = "auto",
    # devices = 1,
    # strategy = "ddp" if config['trainer']['ddp'] else None,
    # log_every_n_steps = 200,
    logger = False,
    callbacks = [custom_checkpoint_callback]
)

trainer.fit(lightning_model, train_dataloader, val_dataloader)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type                 | Params | Mode 
-------------------------------------------------------
0 | model | PeftModelForCausalLM | 833 M  | train
-------------------------------------------------------
4.1 M     Trainable params
828 M     Non-trainable params
833 M     Total params
3,332.476 Total estimated model params size (MB)
1572      Modules in train mode
880       Modules in eval mode


self.lr: 3e-06
Sanity Checking: |          | 0/? [00:00<?, ?it/s]